# ANZSIC Company Classifier

Classifies company names to 4-digit ANZSIC codes using a hybrid approach:

1. **Embedding shortlist** — retrieves top-10 candidate ANZSIC codes via cosine similarity using `all-MiniLM-L6-v2`
2. **LLM reranking** — uses Phi-3-mini (quantized, CPU) to pick the best match from the candidates

The embedding stage is fast but company names lack descriptive content. The LLM stage uses world knowledge to resolve well-known companies (e.g. "Commonwealth Bank" → Banking).

## Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import re
from transformers import AutoTokenizer, AutoModel
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
from pathlib import Path

print(f'torch {torch.__version__} | transformers OK | llama_cpp OK')

## Download Phi-3-mini GGUF Model

Downloads the Q4_K_M quantized model (~2.4 GB) on first run. Subsequent runs skip the download.

In [ ]:
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

GGUF_REPO = 'microsoft/Phi-3-mini-4k-instruct-gguf'
GGUF_FILE = 'Phi-3-mini-4k-instruct-q4.gguf'
GGUF_PATH = MODEL_DIR / GGUF_FILE

if not GGUF_PATH.exists():
    print(f'Downloading {GGUF_FILE} (~2.4 GB)...')
    hf_hub_download(
        repo_id=GGUF_REPO,
        filename=GGUF_FILE,
        local_dir=str(MODEL_DIR),
    )
    print('Download complete.')
else:
    print(f'Model already downloaded: {GGUF_PATH}')

## Load ANZSIC Reference Data

In [ ]:
anzsic = pd.read_parquet('ANZSIC/ANZSIC.parquet')

# Division titles for enriching class-level embeddings
anzsic_divs = (
    anzsic[anzsic['level'] == 'division']
    .set_index('code')['title']
    .to_dict()
)

# Filter to class-level codes (4-digit) — these are the classification targets
anzsic_classes = (
    anzsic[anzsic['level'] == 'class'][['code', 'title', 'division_code']]
    .reset_index(drop=True)
)

# Enriched text for better embeddings: "Division Title - Class Title"
anzsic_classes['embed_text'] = (
    anzsic_classes['division_code'].map(anzsic_divs) + ' - ' + anzsic_classes['title']
)

print(f'ANZSIC classes: {len(anzsic_classes)} entries')
print(f'Divisions: {len(anzsic_divs)}')
anzsic_classes.head()

## Embedding Model Setup

Same `all-MiniLM-L6-v2` model used in the ANZSCO classification pipeline.

In [ ]:
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
DEVICE = 'cpu'
BATCH_SIZE = 256
MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
embed_model = AutoModel.from_pretrained(EMBED_MODEL).to(DEVICE)
embed_model.eval()


def _mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


@torch.no_grad()
def embed_texts(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    texts = [str(t) if pd.notna(t) else '' for t in texts]
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = embed_model(**enc)
        emb = _mean_pool(out.last_hidden_state, enc['attention_mask'])
        emb = F.normalize(emb, p=2, dim=1)
        all_emb.append(emb.cpu().numpy().astype(np.float32))
    return np.vstack(all_emb)


print('Embedding model loaded.')

## Pre-compute ANZSIC Embeddings

In [ ]:
print('Embedding ANZSIC class titles...')
anzsic_embeddings = embed_texts(anzsic_classes['embed_text'].tolist())
print(f'ANZSIC embeddings shape: {anzsic_embeddings.shape}')

## Load LLM

In [ ]:
llm = Llama(
    model_path=str(GGUF_PATH),
    n_ctx=1024,
    n_threads=4,
    n_gpu_layers=0,
    verbose=False,
)
print('LLM loaded.')

## Classification Function

Two-stage hybrid classifier:
1. Embedding retrieval → top-10 ANZSIC candidates
2. LLM reranking → pick the best match (skipped if embedding confidence is very high)

In [ ]:
TOP_K = 10
SIMILARITY_THRESHOLD = 0.85


def classify_company_to_anzsic(company_name: str) -> dict:
    """Classify a company name to a 4-digit ANZSIC class code."""

    # Stage 1: Embedding retrieval
    query_emb = embed_texts([company_name])
    similarities = (query_emb @ anzsic_embeddings.T).flatten()
    top_k_idx = np.argsort(similarities)[-TOP_K:][::-1]
    top_k_scores = similarities[top_k_idx]
    candidates = anzsic_classes.iloc[top_k_idx]

    # Fast path: if embedding confidence is very high, skip LLM
    if top_k_scores[0] >= SIMILARITY_THRESHOLD:
        best = candidates.iloc[0]
        return {
            'anzsic_code': best['code'],
            'anzsic_title': best['title'],
            'confidence': float(top_k_scores[0]),
            'method': 'embedding',
        }

    # Stage 2: LLM reranking
    options = '\n'.join(
        f"  {row['code']}: {row['title']} (Division: {anzsic_divs.get(row['division_code'], '')})"
        for _, row in candidates.iterrows()
    )

    prompt = (
        f'You are an Australian industry classification expert. '
        f'Given a company name, select the most appropriate ANZSIC code from the candidates below.\n\n'
        f'Company: {company_name}\n\n'
        f'Candidate ANZSIC codes:\n{options}\n\n'
        f'Respond with ONLY the 4-digit ANZSIC code that best matches this company. No explanation.'
    )

    response = llm(prompt, max_tokens=10, temperature=0.0, stop=['\n'])
    predicted_code = response['choices'][0]['text'].strip()

    # Validate: extract 4-digit code from response
    match = re.search(r'\b(\d{4})\b', predicted_code)
    if match and match.group(1) in candidates['code'].values:
        code = match.group(1)
        title = candidates[candidates['code'] == code]['title'].values[0]
        return {
            'anzsic_code': code,
            'anzsic_title': title,
            'confidence': float(top_k_scores[0]),
            'method': 'llm_rerank',
        }

    # Fallback to embedding top-1 if LLM output is invalid
    best = candidates.iloc[0]
    return {
        'anzsic_code': best['code'],
        'anzsic_title': best['title'],
        'confidence': float(top_k_scores[0]),
        'method': 'embedding_fallback',
    }

## Batch Classification

Deduplicates company names before calling the LLM to avoid redundant inference.

In [ ]:
def classify_batch(company_names: pd.Series) -> pd.DataFrame:
    """Classify a batch of company names. Deduplicates before LLM calls."""
    unique_names = company_names.dropna().unique()
    print(f'Classifying {len(unique_names)} unique company names...')

    results = {}
    for i, name in enumerate(unique_names):
        results[name] = classify_company_to_anzsic(name)
        if (i + 1) % 10 == 0:
            print(f'  {i + 1}/{len(unique_names)} done')

    print(f'  {len(unique_names)}/{len(unique_names)} done')

    # Map back to original series
    records = []
    for name in company_names:
        if pd.isna(name):
            records.append({'anzsic_code': None, 'anzsic_title': None, 'confidence': None, 'method': None})
        else:
            records.append(results[name])
    return pd.DataFrame(records)

## Evaluation

Classify employer names from the synthetic dataset and compare against ground truth ANZSIC codes.

In [ ]:
employees = pd.read_csv(
    'Test Data/synthetic_employees.csv',
    dtype={'employer_industry_anzsic': str, 'host_employer_industry_anzsic': str},
)

# Zero-pad ANZSIC codes to 4 digits
employees['employer_industry_anzsic'] = (
    employees['employer_industry_anzsic']
    .apply(lambda x: str(int(float(x))).zfill(4) if pd.notna(x) and x != 'nan' else None)
)

print(f'Employees: {len(employees)} rows')
print(f'Unique employer names: {employees["employer_name"].nunique()}')

In [ ]:
predictions = classify_batch(employees['employer_name'])

employees['predicted_anzsic_code'] = predictions['anzsic_code'].values
employees['predicted_anzsic_title'] = predictions['anzsic_title'].values
employees['anzsic_confidence'] = predictions['confidence'].values
employees['anzsic_method'] = predictions['method'].values

## Results

In [ ]:
# Exact match accuracy (4-digit class level)
exact_match = employees['predicted_anzsic_code'] == employees['employer_industry_anzsic']
print(f'Exact match (4-digit):  {exact_match.mean():.1%}  ({exact_match.sum()}/{len(employees)})')

# Group-level accuracy (first 3 digits)
group_match = (
    employees['predicted_anzsic_code'].str[:3] == employees['employer_industry_anzsic'].str[:3]
)
print(f'Group match (3-digit):  {group_match.mean():.1%}  ({group_match.sum()}/{len(employees)})')

# Division-level accuracy (map code to division)
code_to_div = anzsic_classes.set_index('code')['division_code'].to_dict()
pred_div = employees['predicted_anzsic_code'].map(code_to_div)
true_div = employees['employer_industry_anzsic'].map(code_to_div)
div_match = pred_div == true_div
print(f'Division match:         {div_match.mean():.1%}  ({div_match.sum()}/{len(employees)})')

print(f'\nMethod breakdown:')
print(employees['anzsic_method'].value_counts())

In [ ]:
# Per-company results (deduplicated view)
company_results = (
    employees[['employer_name', 'employer_industry_anzsic', 'employer_industry_name',
               'predicted_anzsic_code', 'predicted_anzsic_title', 'anzsic_confidence', 'anzsic_method']]
    .drop_duplicates(subset='employer_name')
    .sort_values('employer_name')
    .reset_index(drop=True)
)

company_results['correct'] = company_results['predicted_anzsic_code'] == company_results['employer_industry_anzsic']

print(f'Per-company accuracy: {company_results["correct"].mean():.1%} ({company_results["correct"].sum()}/{len(company_results)})')
company_results

In [ ]:
# Incorrect predictions
misses = company_results[~company_results['correct']][
    ['employer_name', 'employer_industry_anzsic', 'employer_industry_name',
     'predicted_anzsic_code', 'predicted_anzsic_title', 'anzsic_confidence', 'anzsic_method']
]
print(f'Incorrect predictions: {len(misses)}')
misses

## Ad-hoc Testing

Test with individual company names.

In [ ]:
test_companies = ['Woolworths', 'BHP', 'Telstra', 'Commonwealth Bank', 'Qantas']

for name in test_companies:
    result = classify_company_to_anzsic(name)
    print(f'{name:25s} → {result["anzsic_code"]} {result["anzsic_title"]} ({result["method"]}, conf={result["confidence"]:.4f})')